In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MovieProject").getOrCreate()

In [6]:
# อ่านข้อมูลสำหรับการวิเคราะห์ (Project 1)
df_metadata = spark.read.csv('/content/drive/MyDrive/movies_metadata.csv', header=True, inferSchema=True)

# ตรวจสอบข้อมูลเบื้องต้น
df_metadata.show(5)

+-----+---------------------+--------+--------------------+--------------------+-----+---------+-----------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------+--------------------+--------------------+--------+--------------------+-----------------+
|adult|belongs_to_collection|  budget|              genres|            homepage|   id|  imdb_id|original_language|      original_title|            overview|popularity|         poster_path|production_companies|production_countries|        release_date|             revenue|             runtime|    spoken_languages|  status|             tagline|               title|   video|        vote_average|       vote_count|
+-----+---------------------+--------+--------------------+--------------------+-----+---------+-----------------+--------------------+--------------------+----------+-----

In [7]:
# อ่านข้อมูลสำหรับการทำระบบแนะนำ (Project 2)
df_ratings = spark.read.csv('/content/drive/MyDrive/ratings.csv', header=True, inferSchema=True)
df_ratings.show(5)

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    110|   1.0|1425941529|
|     1|    147|   4.5|1425942435|
|     1|    858|   5.0|1425941523|
|     1|   1221|   5.0|1425941546|
|     1|   1246|   5.0|1425941556|
+------+-------+------+----------+
only showing top 5 rows


# โปรเจค 1

In [10]:
from pyspark.sql.functions import col, desc, expr

# Use try_cast to handle malformed data rows safely
df_cleaned = df_metadata.withColumn("vote_count", expr("try_cast(vote_count as int)")) \
                         .withColumn("vote_average", expr("try_cast(vote_average as float)"))

# Filter out rows where cast failed (NULL) and count is greater than 100
df_filtered = df_cleaned.filter(col("vote_count").isNotNull() & (col("vote_count") > 100))

# Rank top 10 movies
top_movies = df_filtered.select("original_title", "vote_average", "vote_count") \
    .orderBy(desc("vote_average"), desc("vote_count"))

print("Project 1: Data Analysis - Top 10 High Rated Movies")
top_movies.show(10)

Project 1: Data Analysis - Top 10 High Rated Movies
+--------------------+------------+----------+
|      original_title|vote_average|vote_count|
+--------------------+------------+----------+
|Dilwale Dulhania ...|         9.1|       661|
|The Shawshank Red...|         8.5|      8358|
|       The Godfather|         8.5|      6024|
|          君の名は。|         8.5|      1030|
|Dear Zachary: A L...|         8.4|       146|
|     The Dark Knight|         8.3|     12269|
|        Pulp Fiction|         8.3|      8670|
|    Schindler's List|         8.3|      4436|
|            Whiplash|         8.3|      4376|
|    千と千尋の神隠し|         8.3|      3968|
+--------------------+------------+----------+
only showing top 10 rows


In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, desc, row_number, regexp_extract

# 1. ใช้ Regular Expression ดึงชื่อ Genre แรกที่เจอในคอลัมน์ genres
# แพทเทิร์นนี้จะหาคำว่า 'name': '...' แล้วดึงเฉพาะชื่อข้างในออกมา
df_with_genre = df_filtered.withColumn(
    "main_genre",
    regexp_extract(col("genres"), r"'name':\s*'([^']+)'", 1)
)

# 2. กรองแถวที่ไม่มีข้อมูลประเภทหนังออก
df_with_genre = df_with_genre.filter(col("main_genre") != "")

# 3. สร้าง Window เพื่อจัดอันดับหนังที่ดีที่สุด (คะแนนสูง + คนรีวิวเยอะ) แยกตามกลุ่ม main_genre
windowSpec = Window.partitionBy("main_genre").orderBy(desc("vote_average"), desc("vote_count"))

# 4. เลือกเฉพาะอันดับ 1-3 ของแต่ละประเภท
top_3_by_genre = df_with_genre.withColumn("rank", row_number().over(windowSpec)) \
    .filter(col("rank") <= 3) \
    .select("main_genre", "rank", "original_title", "vote_average", "vote_count") \
    .orderBy("main_genre", "rank")

print("Project 1: Analysis - Top 3 Best Alternative Movies by Genre (Cleaned)")
top_3_by_genre.show(30, truncate=False)

Project 1: Analysis - Top 3 Best Alternative Movies by Genre (Cleaned)
+-----------+----+------------------------------------------------+------------+----------+
|main_genre |rank|original_title                                  |vote_average|vote_count|
+-----------+----+------------------------------------------------+------------+----------+
|Action     |1   |七人の侍                                        |8.2         |892       |
|Action     |2   |Band of Brothers                                |8.2         |725       |
|Action     |3   |切腹                                            |8.2         |136       |
|Adventure  |1   |The Empire Strikes Back                         |8.2         |5998      |
|Adventure  |2   |もののけ姫                                      |8.2         |2041      |
|Adventure  |3   |O Auto da Compadecida                           |8.2         |120       |
|Animation  |1   |火垂るの墓                                      |8.2         |974       |
|Animation  |2   |聲の形    

# โปรเจค 2


In [15]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, expr

# 1. จัดการข้อมูลเบื้องต้น (จัดการข้อมูลที่อาจผิดพลาดให้เป็นตัวเลข)
df_ratings_clean = df_ratings.withColumn("userId", expr("try_cast(userId as int)")) \
                             .withColumn("movieId", expr("try_cast(movieId as int)")) \
                             .withColumn("rating", expr("try_cast(rating as float)")) \
                             .dropna()

# 2. แบ่งข้อมูลสำหรับ Train 80% และ Test 20%
(training, test) = df_ratings_clean.randomSplit([0.8, 0.2])

# 3. สร้างและเทรนโมเดล ALS
# coldStartStrategy="drop" เพื่อป้องกัน Error กรณีเจอ User/Movie ใหม่ในตอนทดสอบ
als = ALS(maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId", ratingCol="rating", coldStartStrategy="drop")
model = als.fit(training)

# 4. ทดสอบประสิทธิภาพโมเดล (RMSE)
predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error = {rmse}")

# 5. แสดงผลการแนะนำหนัง 5 เรื่อง สำหรับทุก User
userRecs = model.recommendForAllUsers(5)

# แสดงผลลัพธ์ของ User บางส่วน
print("Project 2: Personalized Movie Recommendations")
userRecs.show(10, truncate=False)

Root-mean-square error = 0.8385512932586139
Project 2: Personalized Movie Recommendations
+------+---------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                          |
+------+---------------------------------------------------------------------------------------------------------+
|1     |[{114980, 16.067652}, {174839, 14.243878}, {126927, 14.203284}, {144706, 14.172093}, {175657, 13.913049}]|
|3     |[{172155, 9.172194}, {164115, 8.696655}, {170605, 8.57068}, {130794, 8.483614}, {114893, 8.334776}]      |
|5     |[{165367, 16.956902}, {69689, 14.023376}, {112577, 14.019733}, {79276, 12.911526}, {145088, 12.648747}]  |
|6     |[{174839, 21.357204}, {174831, 19.831743}, {88674, 18.940117}, {90533, 18.302729}, {131160, 18.15876}]   |
|12    |[{87069, 13.276099}, {105044, 12.89275}, {171561, 11.660902}, {88562, 11.52821}, {130794, 11.2732